# Loading the Olympics Dataset

This notebook illustrates the process of downloading the dataset used for this project from Kagglehub, then loading it into a Neon PostgreSQL database for subsequent SQL querying.

## Tools and libraries

- Python
- Kagglehub
- SQLAlchemy
- python-dotenv
- Psycopg 2

## 1. Download the dataset

The `kagglehub` API is used to directly download the dataset used for this project to disk. In total, two different `.csv` files comprise the dataset: one used as a key to map three-letter abbreviations (National Olympic Committee (NOC) codes) of nations to current nation names (230 entries), and the other containing athlete- and event-specific data for every Olympic event from the inaugural Olympic Games in 1896 to the 2016 Rio de Janeiro Olympics (271,116 entries). Event names, athlete names and IDs, sports categories, medals won, and a variety of other features are present for each entry.

In [1]:
import kagglehub
import os

PATH = kagglehub.dataset_download("heesoo37/120-years-of-olympic-history-athletes-and-results")

os.listdir(PATH)

['noc_regions.csv', 'athlete_events.csv']

## 2. Load into Neon PostgreSQL database

SQLAlchemy is used to connect and load both `.csv` files into Neon via PostgreSQL's
`COPY` command. Important notes about the original dataset:
- Both `.csv` files use the literal string `"NA"` as their null marker (not an empty cell), so `COPY` needs `NULL 'NA'` explicitly.
- `noc_regions.csv` uses bare `\r` line endings, which Postgres's `COPY` doesn't accept (`\n`/`\r\n` only), so this data is normalized in a clean temp file before loading.

Loading this data into Neon requires a `.env` file in this directory containing the Neon database URL used for this project.

#### Connect to the Neon database

Use `load_dotenv` to obtain the connection string for this project's Neon database and establish a connection via the `create_engine` function from `sqlalchemy`.

In [2]:
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Fetch the connection string for this project's Neon database
# .env is not committed for this project; this cell will result in error for users
# who are not the repository owner (unless .env is configured for the new user)

load_dotenv()
engine = create_engine(os.environ["DATABASE_URL"])

# Establish a connection to the Neon database

with engine.connect() as conn:
    result = conn.execute(text("select version()"))
    rows = result.fetchall()
    print(rows[0][0])

PostgreSQL 18.4 (c9a59a4) on aarch64-unknown-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


#### Create the table structures in Neon

The PostgreSQL `COPY` command loads data into an existing table but does not create the table structure itself, so the schema is defined manually in Neon before loading, ensuring each column gets the correct type. `sqlalchemy` is used to execute this table-creation SQL.

In [3]:
# Column order in each table matches the CSV column order exactly, since the COPY command 
# maps by position.

build_tables = """
DROP TABLE IF EXISTS athlete_events;
DROP TABLE IF EXISTS noc_regions;

CREATE TABLE noc_regions (
    noc        TEXT PRIMARY KEY,
    region     TEXT,
    notes      TEXT
);

CREATE TABLE athlete_events (
    id         INTEGER,   
    name       TEXT,
    sex        TEXT,
    age        INTEGER,
    height     NUMERIC,
    weight     NUMERIC,
    team       TEXT,       
    noc        TEXT,
    games      TEXT,
    year       INTEGER,
    season     TEXT,
    city       TEXT,
    sport      TEXT,
    event      TEXT,
    medal      TEXT       
);
"""

with engine.begin() as conn:            # always commit before connection closes
    conn.execute(text(build_tables))

#### Clean `noc_regions` file prior to loading into Neon

For reasons described above, the line endings for `noc_regions.csv` must be changed from `\r` to `\n` (`\r\n` would also work).

In [4]:
# noc_regions.csv has bare \r line endings; normalize to \n in a scratch copy
# before COPY (Postgres's COPY only accepts \n or \r\n).

noc_regions_raw = os.path.join(PATH, "noc_regions.csv")
noc_regions_clean = "/tmp/noc_regions_clean.csv"

with open(noc_regions_raw, "r", encoding="utf-8") as f:
    raw = f.read()

text_normalized = raw.replace("\r\n", "\n").replace("\r", "\n")
with open(noc_regions_clean, "w", encoding="utf-8", newline="\n") as f:
    f.write(text_normalized)

print(f"Normalized noc_regions.csv -> {noc_regions_clean}")

Normalized noc_regions.csv -> /tmp/noc_regions_clean.csv


#### Load the Olympics data into Neon

Now, all data from the Olympics dataset is loaded into Neon via PostgreSQL's `COPY` command, executed through a raw Psycopg 2 cursor's `copy_expert()` method obtained from `engine.raw_connection()`.

In [5]:
csv_paths = {
    "noc_regions": "/tmp/noc_regions_clean.csv",
    "athlete_events": os.path.join(PATH, "athlete_events.csv"),
}

with engine.raw_connection() as raw_conn:       # COPY only works for Postgres, so can't 
                                                # use generic API wrapper ".connect()"
    cur = raw_conn.cursor()
    for table, csv_path in csv_paths.items():
        with open(csv_path, "r", encoding="utf-8") as f:
            cur.copy_expert(f"COPY {table} FROM STDIN WITH (FORMAT csv, HEADER true, NULL 'NA')", f)
        print(f"loaded {table}")
    raw_conn.commit()   # Must manually commit when using raw connection

loaded noc_regions
loaded athlete_events


In [6]:
# One NOC code had no mapped region label; add into noc_regions manually

with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO noc_regions (noc, region, notes)
        VALUES ('SGP', 'Singapore', 'Newer NOC code for Singapore; older rows use SIN')
    """))

In [7]:
# Verify row counts to ensure data was successfully loaded into Neon

with engine.connect() as conn:
    for tbl in list(csv_paths.keys()):
        n = conn.execute(text(f"SELECT COUNT(*) FROM {tbl}")).scalar()
        print(f"{tbl} has {n} rows.")

noc_regions has 231 rows.
athlete_events has 271116 rows.
